In [4]:
import random

import numpy as np
import json
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn import svm
from sklearn.linear_model import SGDClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score
import os
import time
from sklearn.metrics import confusion_matrix, cohen_kappa_score
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, cohen_kappa_score
from sklearn.metrics import ConfusionMatrixDisplay

class Results:
    accuracy = pd.DataFrame()
    precision = pd.DataFrame()
    recall = pd.DataFrame()
    f1 = pd.DataFrame()
    kappa = pd.DataFrame()
    train_accuracy = pd.DataFrame()
    train_precision = pd.DataFrame()
    train_recall = pd.DataFrame()
    train_f1 = pd.DataFrame()
    train_kappa = pd.DataFrame()
    cv_mean = pd.DataFrame()
    cv_stddev = pd.DataFrame()
    train_error_rate = pd.DataFrame()
    gap = pd.DataFrame()
    error_rate = pd.DataFrame()
    train_time = pd.DataFrame()
    infer_time = pd.DataFrame()

    accuracy["Setup"] = ""
    precision["Setup"] = ""
    recall["Setup"] = ""
    f1["Setup"] = ""
    cv_mean["Setup"] = ""
    cv_stddev["Setup"] = ""
    kappa["Setup"] = ""
    error_rate["Setup"] = ""
    train_time["Setup"] = ""
    infer_time["Setup"] = ""
    train_accuracy["Setup"] = ""
    train_precision["Setup"] = ""
    train_recall["Setup"] = ""
    train_f1["Setup"] = ""
    train_kappa["Setup"] = ""
    train_error_rate["Setup"] = ""
    gap["Setup"] = ""

    accuracy.set_index('Setup')
    precision.set_index('Setup')
    recall.set_index('Setup')
    f1.set_index('Setup')
    cv_mean.set_index('Setup')
    cv_stddev.set_index('Setup')
    kappa.set_index('Setup')
    error_rate.set_index('Setup')
    train_time.set_index('Setup')
    infer_time.set_index('Setup')
    train_accuracy.set_index('Setup')
    train_precision.set_index('Setup')
    train_recall.set_index('Setup')
    train_f1.set_index('Setup')
    train_kappa.set_index('Setup')
    train_error_rate.set_index('Setup')
    gap.set_index('Setup')

    def addModel(self, name):
        self.accuracy[name] = ""
        self.precision[name] = ""
        self.recall[name] = ""
        self.f1[name] = ""
        self.cv_mean[name] = ""
        self.cv_stddev[name] = ""
        self.kappa[name] = ""
        self.error_rate[name] = ""
        self.train_time[name] = ""
        self.infer_time[name] = ""
        self.train_accuracy[name] = ""
        self.train_precision[name] = ""
        self.train_recall[name] = ""
        self.train_f1[name] = ""
        self.train_kappa[name] = ""
        self.train_error_rate[name] = ""
        self.gap[name] = ""

    def addResults(self, _acc, _precision, _recall, _f1, _cv_mean, _cv_stddev, _kappa, _error_rate, _train_time, _infer_time, _train_acc, _train_precision, _train_recall, _train_f1, _train_kappa, _train_error_rate, _gap):
        self.accuracy.loc[len(self.accuracy)] = _acc
        self.precision.loc[len(self.precision)] = _precision
        self.recall.loc[len(self.recall)] = _recall
        self.f1.loc[len(self.f1)] = _f1
        self.cv_mean.loc[len(self.cv_mean)] = _cv_mean
        self.cv_stddev.loc[len(self.cv_stddev)] = _cv_stddev
        self.kappa.loc[len(self.kappa)] = _kappa
        self.error_rate.loc[len(self.error_rate)] = _error_rate
        self.train_time.loc[len(self.train_time)] = _train_time
        self.infer_time.loc[len(self.infer_time)] = _infer_time
        self.train_accuracy.loc[len(self.accuracy)] = _train_acc
        self.train_precision.loc[len(self.precision)] = _train_precision
        self.train_recall.loc[len(self.recall)] = _train_recall
        self.train_f1.loc[len(self.f1)] = _train_f1
        self.train_kappa.loc[len(self.kappa)] = _train_kappa
        self.train_error_rate.loc[len(self.error_rate)] = _train_error_rate
        self.gap.loc[len(self.gap)] = _gap

    def printCSV(self, output_folder="Classification"):
      # Cria a pasta com o subdiretório
        full_path = os.path.join("Classification", output_folder)
        os.makedirs(full_path, exist_ok=True)
        self.accuracy.to_csv(f'{full_path}/classification_all_accuracy.csv', index=False, float_format='%.3f')
        self.precision.to_csv(f'{full_path}/classification_all_precision.csv', index=False, float_format='%.3f')
        self.recall.to_csv(f'{full_path}/classification_all_recall.csv', index=False, float_format='%.3f')
        self.f1.to_csv(f'{full_path}/classification_all_f1.csv', index=False, float_format='%.3f')
        self.cv_mean.to_csv(f'{full_path}/classification_all_cv_mean.csv', index=False, float_format='%.3f')
        self.cv_stddev.to_csv(f'{full_path}/classification_all_cv_stddev.csv', index=False, float_format='%.3f')
        self.kappa.to_csv(f'{full_path}/classification_all_kappa.csv', index=False, float_format='%.3f')
        self.error_rate.to_csv(f'{full_path}/classification_all_error_rate.csv', index=False, float_format='%.3f')
        self.train_time.to_csv(f'{full_path}/classification_all_train_time.csv', index=False, float_format='%.3f')
        self.infer_time.to_csv(f'{full_path}/classification_all_infer_time.csv', index=False, float_format='%.3f')

def initialize_data_frame():
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 2000)
    sns.set(rc={'figure.figsize':(19.20, 10.80)})


def get_dungeon_data_frame():
    with open('backup.json') as data_file:
        json_data = json.load(data_file)
    return pd.DataFrame(json_data['__collections__']['dungeons']).T


def get_user_data_frame():
    with open('backup.json') as data_file:
        json_data = json.load(data_file)
    return pd.DataFrame(json_data['__collections__']['users']).T


def drop_invalid_rows(_data_frame):
    _data_frame.drop(_data_frame[_data_frame['UniqueRoomsEntered'] == 0].index, inplace=True)
    return _data_frame


def drop_unused_columns(_data_frame):
    _data_frame.drop('__collections__', inplace=True, axis=1)
    _data_frame = _data_frame.drop(list(_data_frame.filter(regex='^Total.*Quests')), axis=1)
    _data_frame.drop('CompletedQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedKillQuests', inplace=True, axis=1)
    _data_frame.rename(columns={'CompletedDamageQuests': 'CompletedKillQuests'}, inplace=True)
    _data_frame.drop('TreasuresCollected', inplace=True, axis=1)
    _data_frame.drop('TotalNpcs', inplace=True, axis=1)
    _data_frame.drop('WeaponName', inplace=True, axis=1)
    _data_frame.drop('HeatMap', inplace=True, axis=1)
    return _data_frame


def get_percentages(_data_frame):
    _data_frame['LockUsage'] = _data_frame['LocksOpened'].div(_data_frame['TotalLocks'].values)*100
    _data_frame['KeyCollection'] = _data_frame['KeysCollected'].div(_data_frame['TotalKeys'].values)*100
    _data_frame['MapCompletion'] = _data_frame['UniqueRoomsEntered'].div(_data_frame['TotalRooms'].values)*100
    _data_frame['RoomRevisitRate'] = _data_frame['RoomsEntered'].div(_data_frame['TotalRooms'].values)*100
    _data_frame['EnemyKillRate'] = _data_frame['EnemiesKilled'].div(_data_frame['TotalEnemies'].values)*100
    _data_frame['TotalAttempts'] = _data_frame['TotalWins'] + _data_frame['TotalDeaths'].values
    _data_frame['TotalAttempts'] = _data_frame['TotalAttempts'].astype(float)
    _data_frame['TotalWins'] = _data_frame['TotalWins'].astype(float)
    _data_frame['Success Rate'] = _data_frame['TotalAttempts'].div(_data_frame['TotalWins'].values)*100
    _data_frame.replace([np.inf, -np.inf], 0, inplace=True)
    return _data_frame


def drop_processed_columns(_data_frame):
    _data_frame.drop('LocksOpened', inplace=True, axis=1)
    _data_frame.drop('TotalLocks', inplace=True, axis=1)
    _data_frame.drop('KeysCollected', inplace=True, axis=1)
    _data_frame.drop('TotalKeys', inplace=True, axis=1)
    _data_frame.drop('EnemiesKilled', inplace=True, axis=1)
    _data_frame.drop('TotalEnemies', inplace=True, axis=1)
    _data_frame.drop('TotalAttempts', inplace=True, axis=1)
    _data_frame.drop('TotalWins', inplace=True, axis=1)
    _data_frame.drop('TotalDeaths', inplace=True, axis=1)
    _data_frame.drop('NpcsInteracted', inplace=True, axis=1)
    _data_frame.drop('RoomsEntered', inplace=True, axis=1)
    _data_frame.drop('TotalRooms', inplace=True, axis=1)
    _data_frame.drop('UniqueRoomsEntered', inplace=True, axis=1)
    _data_frame.drop('TotalTreasure', inplace=True, axis=1)
    return _data_frame


def split_preform_answers(_data_frame):
    _data_frame[['A', 'C', 'I', 'M']] = _data_frame['PreFormAnswers'].str.split(',', expand=True)
    _data_frame['A'] = _data_frame['A'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    _data_frame['C'] = _data_frame['C'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    _data_frame['I'] = _data_frame['I'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    _data_frame['M'] = _data_frame['M'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    return _data_frame


def drop_quest_data(_data_frame):
    _data_frame = _data_frame.drop(list(_data_frame.filter(regex='Quest')), axis=1)
    return _data_frame


def drop_player_data(_data_frame):
    _data_frame.drop('A', inplace=True, axis=1)
    _data_frame.drop('C', inplace=True, axis=1)
    _data_frame.drop('I', inplace=True, axis=1)
    _data_frame.drop('M', inplace=True, axis=1)
    _data_frame.drop('PostFormAnswers', inplace=True, axis=1)
    _data_frame.drop('PlayerId', inplace=True, axis=1)
    return _data_frame


def drop_gameplay_data(_data_frame):
    _data_frame.drop('TimeToFinish', inplace=True, axis=1)
    _data_frame.drop('LockUsage', inplace=True, axis=1)
    _data_frame.drop('KeyCollection', inplace=True, axis=1)
    _data_frame.drop('MapCompletion', inplace=True, axis=1)
    _data_frame.drop('EnemyKillRate', inplace=True, axis=1)
    return _data_frame


def drop_terminal_quest_data(_data_frame):
    _data_frame.drop('CompletedReportQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedListenQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedExploreQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedExchangeQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedKillQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedGiveQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedGoToQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedGatherQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedReadQuests', inplace=True, axis=1)
    return _data_frame


def augment_dataframe_with_scaling(df, y, n, scale_factor):
    """
    Augments the DataFrame by adding `n` new rows, scaling numeric columns by a given factor.
    The new rows inherit the same 'y' values as their corresponding sampled rows.

    Parameters:
        df (pd.DataFrame): Original DataFrame with numeric columns and a 'y' column.
        n (int): Number of new rows to generate.
        scale_factor (float): The factor by which numeric columns will be scaled.

    Returns:
        pd.DataFrame: DataFrame with the original and new rows.
    """
    # Identify numeric columns, excluding 'y'
    numeric_columns = df.select_dtypes(include=['number']).columns.difference(y)
    # Randomly sample `n` rows from the original DataFrame
    sampled_rows = df.sample(n=n, replace=True).reset_index(drop=True)

    # Scale numeric columns
    for col in numeric_columns:
        sampled_rows[col] *= scale_factor

    # Append the scaled rows to the original DataFrame
    augmented_df = pd.concat([df, sampled_rows], ignore_index=True)

    return augmented_df


def augment_data_random(_df, y, n):
    """
    Augments the DataFrame by adding `n` new rows.
    - Adds noise to numerical columns using a normal distribution.
    - Retains the same 'y' values for the newly added rows as their corresponding sampled rows.

    Parameters:
        _df (pd.DataFrame): Original DataFrame with numeric columns and a 'y' column.
        y (list): the column(s) that will have their value retained
        n (int): Number of new rows to generate.

    Returns:
        pd.DataFrame: DataFrame with the original and new rows.
    """
    # Identify float and integer columns, excluding 'y'
    float_columns = _df.select_dtypes(include=['float']).columns.difference(y)
    int_columns = _df.select_dtypes(include=['integer']).columns.difference(y)

    # Randomly sample `n` rows from the original DataFrame
    sampled_rows = _df.sample(n=n, replace=True).reset_index(drop=True)

    # Add noise to float columns
    for col in float_columns:
        if random.randint(1, 100) > 50:
            std = _df[col].std()
            noise = np.random.normal(loc=0, scale=std, size=n)
            sampled_rows[col] += noise

    for col in int_columns:
        if random.randint(1, 100) > 50:
            std = _df[col].std()
            noise = np.random.normal(loc=0, scale=std, size=n)
            sampled_rows[col] += noise

    # Ensure integer columns remain integers
    sampled_rows[int_columns] = sampled_rows[int_columns].round().astype(int)

    # Append the sampled (augmented) rows to the original DataFrame
    augmented_df = pd.concat([_df, sampled_rows], ignore_index=True)
    return augmented_df


def classify(_df, models, output_folder):
    _df.drop('PostFormAnswers', axis=1, inplace=True)
    _df.drop('PlayerId', axis=1, inplace=True)
    _df.drop('InputProfile', axis=1, inplace=True)
    _df.drop('LevelName', axis=1, inplace=True)
    _df['PreFormAnswers'] = _df['PreFormAnswers'].apply(lambda x: 'A=' + str(x[0]) + ", C=" + str(x[1]) + ", I=" + str(x[2]) +
                                                                  ", M=" + str(x[3]))
    _df = split_preform_answers(_df)
    n = _df.shape[0]

    _df.describe().to_csv('descriptive_statistics.csv')


    _dfnoise = augment_data_random(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n)
    _dfscale = augment_dataframe_with_scaling(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)
    _dfnoise2 = augment_data_random(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], 2*n)
    _dfscale2 = augment_dataframe_with_scaling(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)
    _dfscale2 = augment_dataframe_with_scaling(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.95)
    _dfmixed = augment_data_random(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n)
    _dfmixed = pd.concat([_dfmixed, augment_dataframe_with_scaling(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)], ignore_index=True)
    _dfmixed10 = augment_data_random(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], 6*n)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)], ignore_index=True)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.95)], ignore_index=True)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.10)], ignore_index=True)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.9)], ignore_index=True)

    dataframes = [#_df, _dfnoise, _dfscale, _dfnoise2, _dfscale2, _dfmixed,
                  _dfmixed10]
    experiment_names = [#'Baseline', '1xN', '1xS-1_05', '2xN', '2xS-1_05+0_95', '1xS-1_05+1xN',
                        '6xN+4xS-1_05+0_95+1_1+0_9']

    strategies = [#'Original', '1xNoise', '1xScale', '2xN', '2xScale', '1xN+1xS',
                  '6xN+4xS']

    results = Results()
    # LIMPA COMPLETAMENTE TODOS OS RESULTADOS ANTERIORES

    results.accuracy = pd.DataFrame(columns=["Setup"])
    results.precision = pd.DataFrame(columns=["Setup"])
    results.recall = pd.DataFrame(columns=["Setup"])
    results.f1 = pd.DataFrame(columns=["Setup"])
    results.kappa = pd.DataFrame(columns=["Setup"])

    results.train_accuracy = pd.DataFrame(columns=["Setup"])
    results.train_precision = pd.DataFrame(columns=["Setup"])
    results.train_recall = pd.DataFrame(columns=["Setup"])
    results.train_f1 = pd.DataFrame(columns=["Setup"])
    results.train_kappa = pd.DataFrame(columns=["Setup"])

    results.cv_mean = pd.DataFrame(columns=["Setup"])
    results.cv_stddev = pd.DataFrame(columns=["Setup"])

    results.train_error_rate = pd.DataFrame(columns=["Setup"])
    results.error_rate = pd.DataFrame(columns=["Setup"])
    results.gap = pd.DataFrame(columns=["Setup"])

    results.train_time = pd.DataFrame(columns=["Setup"])
    results.infer_time = pd.DataFrame(columns=["Setup"])

    for model in models:
        results.addModel(model)

    for i in range(len(strategies)):
        _experimentdf = dataframes[i]

        profiles = _experimentdf['PreFormAnswers']
        _experimentdf.drop('PreFormAnswers', axis=1, inplace=True)
        _experimentdf.drop('A', axis=1, inplace=True)
        _experimentdf.drop('C', axis=1, inplace=True)
        _experimentdf.drop('I', axis=1, inplace=True)
        _experimentdf.drop('M', axis=1, inplace=True)
        _experimentdf.drop('Success Rate', axis=1, inplace=True)
        X_train, X_test, y_train, y_test = train_test_split(_experimentdf, profiles, test_size=0.3, random_state=42)

        min_max_scaler = MinMaxScaler()
        X_train = min_max_scaler.fit_transform(X_train)

        experiment_name = experiment_names[i]

        train_accuracy = [experiment_name]
        train_precision = [experiment_name]
        train_recall = [experiment_name]
        train_f1score = [experiment_name]
        train_kappa = [experiment_name]
        train_error_rate = [experiment_name]
        accuracy = [experiment_name]
        precision = [experiment_name]
        recall = [experiment_name]
        f1score = [experiment_name]
        cv_mean = [experiment_name]
        cv_stddev = [experiment_name]
        kappa = [experiment_name]
        error_rate = [experiment_name]
        train_time = [experiment_name]
        infer_time = [experiment_name]
        gap = [experiment_name]

        for name, model in models.items():
            start_train = time.perf_counter()
            model.fit(X_train, y_train)
            train_pred = model.predict(X_train)
            end_train = time.perf_counter()
            #train_time = end_train - start_train
            #print(f"Tempo de treino: {train_time:0.4f} segundos")

            start_pred = time.perf_counter()
            pred = model.predict(X_test)
            end_pred = time.perf_counter()
            #infer_time = end_pred - start_pred
            #print(f"Tempo de treino: {infer_time:0.4f} segundos")

            train_accuracy.append(accuracy_score(y_train, train_pred))
            train_precision.append(precision_score(y_train, train_pred, average='weighted'))
            train_recall.append(recall_score(y_train, train_pred, average='weighted'))
            train_f1score.append(f1_score(y_train, train_pred, average='weighted'))
            train_error_rate.append(1 - accuracy_score(y_train, train_pred))
            train_kappa.append(cohen_kappa_score(y_train, train_pred))

            accuracy.append(accuracy_score(y_test, pred))
            precision.append(precision_score(y_test, pred, average='weighted'))
            recall.append(recall_score(y_test, pred, average='weighted'))
            f1score.append(f1_score(y_test, pred, average='weighted'))
            error_rate.append(1 - accuracy_score(y_test, pred))
            kappa.append(cohen_kappa_score(y_test, pred))
            gap.append(accuracy_score(y_train, train_pred) - accuracy_score(y_test, pred))
            train_time.append(end_train - start_train)
            infer_time.append(end_pred - start_pred)

            # ---------- Matriz de confusão ----------
            cm = confusion_matrix(y_test, pred)
            # Cria a pasta com output_folder no caminho
            full_path = os.path.join("Classification/", output_folder)
            os.makedirs(full_path, exist_ok=True)

            with open(f"{full_path}/confusion_matrices.txt", "a") as f:
              f.write(f"\n=== Experimento: {experiment_name} | Modelo: {model} ===\n")
              f.write(str(cm))
              f.write("\n")

            scores = cross_val_score(model, _experimentdf, profiles, cv=5)
            cv_mean.append(scores.mean())
            cv_stddev.append(scores.std())

            print("\n==============================")
            print(f"Experimento: {experiment_name}")
            print(f"Modelo: {name}")
            print("------------------------------")

            print("TRAIN")
            print(f"Accuracy: {train_accuracy[-1]:.4f}")
            print(f"Precision: {train_precision[-1]:.4f}")
            print(f"Recall: {train_recall[-1]:.4f}")
            print(f"F1: {train_f1score[-1]:.4f}")
            print(f"Kappa: {train_kappa[-1]:.4f}")
            print(f"Error Rate: {train_error_rate[-1]:.4f}")

            print("\nTEST")
            print(f"Accuracy: {accuracy[-1]:.4f}")
            print(f"Precision: {precision[-1]:.4f}")
            print(f"Recall: {recall[-1]:.4f}")
            print(f"F1: {f1score[-1]:.4f}")
            print(f"Kappa: {kappa[-1]:.4f}")
            print(f"Error Rate: {error_rate[-1]:.4f}")
            print(f"Gap (overfitting): {gap[-1]:.4f}")

            print("\nCV")
            print(f"Mean: {cv_mean[-1]:.4f}")
            print(f"Std: {cv_stddev[-1]:.4f}")

            print("------------------------------")
            print("Model params:")
            print(model.get_params())
            print("==============================\n")

            fig, ax = plt.subplots(figsize=(8, 6))
            disp = ConfusionMatrixDisplay(confusion_matrix=cm)
            disp.plot(ax=ax)

            plt.title("Confusion Matrix")

            # Cria a pasta com output_folder no caminho
            full_path = os.path.join("Classification/", output_folder, "images")
            os.makedirs(full_path, exist_ok=True)

            # Salva a figura na subpasta
            plt.savefig(
                f"{full_path}/confusion_matrix_{experiment_name}_{model}.png",
                dpi=300,
                bbox_inches='tight'
            )

            plt.show()
            plt.close(fig)



        results.addResults(
            accuracy, precision, recall, f1score,
            cv_mean, cv_stddev,
            kappa, error_rate,
            train_time, infer_time,
            train_accuracy, train_precision, train_recall,
            train_f1score, train_kappa, train_error_rate,
            gap
        )
    results.printCSV(output_folder)

def reset_results():
    Results.accuracy = pd.DataFrame()
    Results.precision = pd.DataFrame()
    Results.recall = pd.DataFrame()
    Results.f1 = pd.DataFrame()
    Results.kappa = pd.DataFrame()
    Results.train_accuracy = pd.DataFrame()
    Results.train_precision = pd.DataFrame()
    Results.train_recall = pd.DataFrame()
    Results.train_f1 = pd.DataFrame()
    Results.train_kappa = pd.DataFrame()
    Results.cv_mean = pd.DataFrame()
    Results.cv_stddev = pd.DataFrame()
    Results.train_error_rate = pd.DataFrame()
    Results.gap = pd.DataFrame()
    Results.error_rate = pd.DataFrame()
    Results.train_time = pd.DataFrame()
    Results.infer_time = pd.DataFrame()

def main(models, folder):
    initialize_data_frame()
    dungeon_frame = get_dungeon_data_frame()
    user_frame = get_user_data_frame()
    dungeon_frame = pd.merge(dungeon_frame, user_frame[['PlayerId', 'PreFormAnswers']], on=['PlayerId'], how='left')
    dungeon_frame = drop_invalid_rows(dungeon_frame)
    dungeon_frame = drop_unused_columns(dungeon_frame)
    dungeon_frame = get_percentages(dungeon_frame)
    dungeon_frame = drop_processed_columns(dungeon_frame)
    classify(dungeon_frame, models,folder)

def teste():
    initialize_data_frame()

    dungeon_frame = get_dungeon_data_frame()
    user_frame = get_user_data_frame()

    # 🔥 CONVERTE PARA TUPLA AQUI
    profiles = user_frame['PreFormAnswers'].apply(tuple)

    print("Combinações únicas:")
    print(sorted(profiles.unique()))

    print("\nQuantidade de combinações únicas:")
    print(profiles.nunique())

    print("\nFrequência de cada combinação:")
    print(profiles.value_counts())

    dungeon_frame = pd.merge(
        dungeon_frame,
        user_frame[['PlayerId', 'PreFormAnswers']],
        on=['PlayerId'],
        how='left'
    )


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, cohen_kappa_score
from sklearn.model_selection import cross_val_score
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import LabelEncoder

ModuleNotFoundError: No module named 'torch'

In [ ]:
class CNN_LSTM_Hybrid(BaseEstimator, ClassifierMixin):
    """CNN para extrair features locais + LSTM para capturar dependências"""

    def __init__(self,
                 cnn_filters=[16, 32],
                 cnn_kernels=[3, 3],
                 cnn_dropout=0.2,
                 lstm_hidden=32,
                 lstm_layers=1,
                 lstm_dropout=0.2,
                 fc_hidden=(16,),
                 input_size=None,
                 output_size=8,
                 solver='adam',           # ← ADICIONADO
                 learning_rate=0.001,
                 momentum=0.9,            # ← ADICIONADO
                 epochs=100,
                 batch_size=32,
                 random_state=42,
                 device='cpu'):

        self.cnn_filters = cnn_filters
        self.cnn_kernels = cnn_kernels
        self.cnn_dropout = cnn_dropout
        self.lstm_hidden = lstm_hidden
        self.lstm_layers = lstm_layers
        self.lstm_dropout = lstm_dropout
        self.fc_hidden = fc_hidden
        self.input_size = input_size
        self.output_size = output_size
        self.solver = solver              # ← NOVO
        self.learning_rate = learning_rate
        self.momentum = momentum          # ← NOVO
        self.epochs = epochs
        self.batch_size = batch_size
        self.random_state = random_state
        self.device = device if torch.cuda.is_available() else 'cpu'

        self.model = None
        self.optimizer = None
        self.criterion = None
        self.label_encoder = LabelEncoder()
        self.scaler = MinMaxScaler()

    def _build_model(self):
        class CNN_LSTM_Model(nn.Module):
            def __init__(self, input_size, cnn_filters, cnn_kernels, cnn_dropout,
                         lstm_hidden, lstm_layers, lstm_dropout, fc_hidden, output_size):
                super(CNN_LSTM_Model, self).__init__()

                # CNN Layers
                conv_layers = []
                prev_channels = 1

                for filters, kernel in zip(cnn_filters, cnn_kernels):
                    conv_layers.append(
                        nn.Conv1d(prev_channels, filters, kernel, padding='same')
                    )
                    conv_layers.append(nn.BatchNorm1d(filters))
                    conv_layers.append(nn.ReLU())
                    conv_layers.append(nn.MaxPool1d(2))
                    conv_layers.append(nn.Dropout(cnn_dropout))
                    prev_channels = filters

                self.conv_layers = nn.Sequential(*conv_layers)

                # Calcular tamanho após CNN
                cnn_out_size = input_size // (2 ** len(cnn_filters))

                # LSTM
                self.lstm = nn.LSTM(
                    input_size=prev_channels,
                    hidden_size=lstm_hidden,
                    num_layers=lstm_layers,
                    batch_first=True,
                    dropout=lstm_dropout if lstm_layers > 1 else 0
                )

                self.lstm_dropout = nn.Dropout(lstm_dropout)

                # Fully connected
                fc_layers = []
                prev_size = lstm_hidden

                for hidden in fc_hidden:
                    fc_layers.append(nn.Linear(prev_size, hidden))
                    fc_layers.append(nn.ReLU())
                    fc_layers.append(nn.Dropout(cnn_dropout))
                    prev_size = hidden

                fc_layers.append(nn.Linear(prev_size, output_size))
                self.fc_layers = nn.Sequential(*fc_layers)

            def forward(self, x):
                # CNN forward
                x = x.unsqueeze(1)  # (batch, 1, features)
                x = self.conv_layers(x)  # (batch, channels, features_cnn)

                # Reshape para LSTM: (batch, seq_len, features)
                x = x.permute(0, 2, 1)  # (batch, features_cnn, channels)

                # LSTM forward
                lstm_out, (hidden, cell) = self.lstm(x)
                last_hidden = hidden[-1]
                x = self.lstm_dropout(last_hidden)

                # Fully connected
                x = self.fc_layers(x)
                return x

        return CNN_LSTM_Model(
            input_size=self.input_size,
            cnn_filters=self.cnn_filters,
            cnn_kernels=self.cnn_kernels,
            cnn_dropout=self.cnn_dropout,
            lstm_hidden=self.lstm_hidden,
            lstm_layers=self.lstm_layers,
            lstm_dropout=self.lstm_dropout,
            fc_hidden=self.fc_hidden,
            output_size=self.output_size
        )

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(y, (pd.Series, pd.DataFrame)):
            y = y.values

        if self.input_size is None:
            self.input_size = X.shape[1]

        X_scaled = self.scaler.fit_transform(X)
        y_encoded = self.label_encoder.fit_transform(y)
        self.output_size = len(self.label_encoder.classes_)

        torch.manual_seed(self.random_state)
        self.model = self._build_model().to(self.device)

        # ============================================================
        # ADAPTADO: Escolher otimizador baseado em self.solver
        # ============================================================
        if self.solver == 'adam':
            self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        elif self.solver == 'sgd':
            self.optimizer = optim.SGD(self.model.parameters(), lr=self.learning_rate, momentum=self.momentum)
        elif self.solver == 'rmsprop':
            self.optimizer = optim.RMSprop(self.model.parameters(), lr=self.learning_rate)
        else:
            self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)

        self.criterion = nn.CrossEntropyLoss()

        X_tensor = torch.FloatTensor(X_scaled).to(self.device)
        y_tensor = torch.LongTensor(y_encoded).to(self.device)

        dataset = TensorDataset(X_tensor, y_tensor)
        dataloader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

        self.model.train()
        for epoch in range(self.epochs):
            total_loss = 0
            for batch_X, batch_y in dataloader:
                self.optimizer.zero_grad()
                outputs = self.model(batch_X)
                loss = self.criterion(outputs, batch_y)
                loss.backward()
                self.optimizer.step()
                total_loss += loss.item()

            if (epoch + 1) % 20 == 0:
                print(f"Epoch {epoch+1}/{self.epochs}, Loss: {total_loss/len(dataloader):.4f}")

        return self

    def predict(self, X):
        if isinstance(X, pd.DataFrame):
            X = X.values

        self.model.eval()
        X_scaled = self.scaler.transform(X)
        X_tensor = torch.FloatTensor(X_scaled).to(self.device)

        with torch.no_grad():
            outputs = self.model(X_tensor)
            _, predicted = torch.max(outputs, 1)

        return self.label_encoder.inverse_transform(predicted.cpu().numpy())

    def predict_proba(self, X):
        if isinstance(X, pd.DataFrame):
            X = X.values

        self.model.eval()
        X_scaled = self.scaler.transform(X)
        X_tensor = torch.FloatTensor(X_scaled).to(self.device)

        with torch.no_grad():
            outputs = torch.softmax(self.model(X_tensor), dim=1)

        return outputs.cpu().numpy()

    def get_params(self, deep=True):
        return {
            'cnn_filters': self.cnn_filters,
            'cnn_kernels': self.cnn_kernels,
            'cnn_dropout': self.cnn_dropout,
            'lstm_hidden': self.lstm_hidden,
            'lstm_layers': self.lstm_layers,
            'lstm_dropout': self.lstm_dropout,
            'fc_hidden': self.fc_hidden,
            'input_size': self.input_size,
            'output_size': self.output_size,
            'solver': self.solver,           # ← NOVO
            'learning_rate': self.learning_rate,
            'momentum': self.momentum,       # ← NOVO
            'epochs': self.epochs,
            'batch_size': self.batch_size,
            'random_state': self.random_state,
            'device': self.device
        }

    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

In [ ]:
models_hybrid = {}

models_hybrid['Hybrid_rmsprop'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='rmsprop', learning_rate=0.001, epochs=500, batch_size=32, random_state=42
)

models_hybrid['Hybrid_adam'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='adam', learning_rate=0.001, epochs=500, batch_size=32, random_state=42
)

models_hybrid['Hybrid_sgd_momentum'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='sgd', learning_rate=0.001, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

models_hybrid['Hybrid_adam_lowLR'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='adam', learning_rate=0.0001, epochs=500, batch_size=32, random_state=42
)

main(models_hybrid, "hybrid_experiments")

/tmp/ipykernel_479/1652769159.py:178: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _data_frame.replace([np.inf, -np.inf], 0, inplace=True)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
models_hybrid_f1 = {}

# ============================================================
# 1º MELHOR: Hybrid_sgd_momentum
# ============================================================

models_hybrid_f1['Hybrid_sgd_30'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='sgd', learning_rate=0.001, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f1['Hybrid_sgd_50'] = CNN_LSTM_Hybrid(
    cnn_filters=[50], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=50, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='sgd', learning_rate=0.001, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

# 2 camadas CNN + 2 camadas LSTM? Na verdade: CNN=[50,25] + LSTM hidden=25
models_hybrid_f1['Hybrid_sgd_50_25'] = CNN_LSTM_Hybrid(
    cnn_filters=[50, 25], cnn_kernels=[3, 3], cnn_dropout=0.0,
    lstm_hidden=25, lstm_layers=2, lstm_dropout=0.0, fc_hidden=(25,),
    solver='sgd', learning_rate=0.001, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

# Regularização CNN
models_hybrid_f1['Hybrid_sgd_reg1e3'] = CNN_LSTM_Hybrid(
    cnn_filters=[50], cnn_kernels=[3], cnn_dropout=0.3,
    lstm_hidden=50, lstm_layers=1, lstm_dropout=0.3, fc_hidden=(),
    solver='sgd', learning_rate=0.001, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f1['Hybrid_sgd_reg5e3'] = CNN_LSTM_Hybrid(
    cnn_filters=[50], cnn_kernels=[3], cnn_dropout=0.5,
    lstm_hidden=50, lstm_layers=1, lstm_dropout=0.5, fc_hidden=(),
    solver='sgd', learning_rate=0.001, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

# ============================================================
# 2º MELHOR: Hybrid_rmsprop
# ============================================================

models_hybrid_f1['Hybrid_rmsprop_30'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='rmsprop', learning_rate=0.001, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f1['Hybrid_rmsprop_50'] = CNN_LSTM_Hybrid(
    cnn_filters=[50], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=50, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='rmsprop', learning_rate=0.001, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f1['Hybrid_rmsprop_50_25'] = CNN_LSTM_Hybrid(
    cnn_filters=[50, 25], cnn_kernels=[3, 3], cnn_dropout=0.0,
    lstm_hidden=25, lstm_layers=2, lstm_dropout=0.0, fc_hidden=(25,),
    solver='rmsprop', learning_rate=0.001, epochs=500, batch_size=32, random_state=42
)

# ============================================================
# 3º MELHOR: Hybrid_adam_lowLR
# ============================================================

models_hybrid_f1['Hybrid_adam_lowLR_30'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='adam', learning_rate=0.0001, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f1['Hybrid_adam_lowLR_50'] = CNN_LSTM_Hybrid(
    cnn_filters=[50], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=50, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='adam', learning_rate=0.0001, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f1['Hybrid_adam_lowLR_50_25'] = CNN_LSTM_Hybrid(
    cnn_filters=[50, 25], cnn_kernels=[3, 3], cnn_dropout=0.0,
    lstm_hidden=25, lstm_layers=2, lstm_dropout=0.0, fc_hidden=(25,),
    solver='adam', learning_rate=0.0001, epochs=500, batch_size=32, random_state=42
)

main(models_hybrid_f1, "hybrid_fase_1")

In [ ]:
models_hybrid_f2 = {}

# ============================================================
# OTIMIZADORES
# ============================================================

models_hybrid_f2['Hybrid_solver_adam'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='adam', learning_rate=0.0001, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f2['Hybrid_solver_sgd'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='sgd', learning_rate=0.001, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f2['Hybrid_solver_rmsprop'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='rmsprop', learning_rate=0.001, epochs=500, batch_size=32, random_state=42
)

# ============================================================
# ARQUITETURAS
# ============================================================

# hidden=10 equivalente
models_hybrid_f2['Hybrid_arch_10'] = CNN_LSTM_Hybrid(
    cnn_filters=[10], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=10, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='adam', learning_rate=0.0001, epochs=500, batch_size=32, random_state=42
)

# hidden=30
models_hybrid_f2['Hybrid_arch_30'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='adam', learning_rate=0.0001, epochs=500, batch_size=32, random_state=42
)

# hidden=50
models_hybrid_f2['Hybrid_arch_50'] = CNN_LSTM_Hybrid(
    cnn_filters=[50], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=50, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='adam', learning_rate=0.0001, epochs=500, batch_size=32, random_state=42
)

# 2 camadas CNN + 2 camadas LSTM (50_25)
models_hybrid_f2['Hybrid_arch_50_25'] = CNN_LSTM_Hybrid(
    cnn_filters=[50, 25], cnn_kernels=[3, 3], cnn_dropout=0.0,
    lstm_hidden=25, lstm_layers=2, lstm_dropout=0.0, fc_hidden=(25,),
    solver='adam', learning_rate=0.0001, epochs=500, batch_size=32, random_state=42
)

# ============================================================
# REGULARIZAÇÃO
# ============================================================

models_hybrid_f2['Hybrid_dropout_0'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='adam', learning_rate=0.0001, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f2['Hybrid_dropout_1e-1'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.1,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.1, fc_hidden=(),
    solver='adam', learning_rate=0.0001, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f2['Hybrid_dropout_1e-1.5'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.2,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.2, fc_hidden=(),
    solver='adam', learning_rate=0.0001, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f2['Hybrid_dropout_1e-1.8'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.3,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.3, fc_hidden=(),
    solver='adam', learning_rate=0.0001, epochs=500, batch_size=32, random_state=42
)

main(models_hybrid_f2, "hybrid_fase_2")

In [ ]:
models_hybrid_f3 = {}

# ============================================================
# 🔥 BASE TOP (SGD)
# ============================================================

models_hybrid_f3['F2_30_sgd'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='sgd', learning_rate=0.001, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f3['F2_30_sgd_momentum95'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='sgd', learning_rate=0.001, momentum=0.95, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f3['F2_50_sgd'] = CNN_LSTM_Hybrid(
    cnn_filters=[50], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=50, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='sgd', learning_rate=0.001, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f3['F2_50_sgd_momentum95'] = CNN_LSTM_Hybrid(
    cnn_filters=[50], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=50, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='sgd', learning_rate=0.001, momentum=0.95, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f3['F2_50_25_sgd'] = CNN_LSTM_Hybrid(
    cnn_filters=[50, 25], cnn_kernels=[3, 3], cnn_dropout=0.0,
    lstm_hidden=25, lstm_layers=2, lstm_dropout=0.0, fc_hidden=(25,),
    solver='sgd', learning_rate=0.001, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f3['F2_50_25_sgd_momentum95'] = CNN_LSTM_Hybrid(
    cnn_filters=[50, 25], cnn_kernels=[3, 3], cnn_dropout=0.0,
    lstm_hidden=25, lstm_layers=2, lstm_dropout=0.0, fc_hidden=(25,),
    solver='sgd', learning_rate=0.001, momentum=0.95, epochs=500, batch_size=32, random_state=42
)

# ============================================================
# ⚖️ CONTROLE (ADAM)
# ============================================================

models_hybrid_f3['F2_30_adam_reg1e-4'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.2,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.2, fc_hidden=(),
    solver='adam', learning_rate=0.0001, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f3['F2_50_adam_reg1e-4'] = CNN_LSTM_Hybrid(
    cnn_filters=[50], cnn_kernels=[3], cnn_dropout=0.2,
    lstm_hidden=50, lstm_layers=1, lstm_dropout=0.2, fc_hidden=(),
    solver='adam', learning_rate=0.0001, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f3['F2_50_25_adam_reg1e-4'] = CNN_LSTM_Hybrid(
    cnn_filters=[50, 25], cnn_kernels=[3, 3], cnn_dropout=0.2,
    lstm_hidden=25, lstm_layers=2, lstm_dropout=0.2, fc_hidden=(25,),
    solver='adam', learning_rate=0.0001, epochs=500, batch_size=32, random_state=42
)

# ============================================================
# 🔥 EXTRA: Regularização para SGD
# ============================================================

models_hybrid_f3['F2_30_sgd_dropout2'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.2,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.2, fc_hidden=(),
    solver='sgd', learning_rate=0.001, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f3['F2_30_sgd_dropout3'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.3,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.3, fc_hidden=(),
    solver='sgd', learning_rate=0.001, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f3['F2_30_sgd_lowLR'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='sgd', learning_rate=0.0001, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

main(models_hybrid_f3, "hybrid_fase_3")

In [ ]:
models_hybrid_f4 = {}

# ============================================================
# 🔥 SGD com arquitetura base (30)
# ============================================================

models_hybrid_f4['F3_30_lr1e-3_dropout0_sgd'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='sgd', learning_rate=0.001, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f4['F3_30_lr5e-4_dropout0_sgd'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='sgd', learning_rate=0.0005, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f4['F3_30_lr1e-4_dropout0_sgd'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.0,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.0, fc_hidden=(),
    solver='sgd', learning_rate=0.0001, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f4['F3_30_lr1e-3_dropout1_sgd'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.1,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.1, fc_hidden=(),
    solver='sgd', learning_rate=0.001, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f4['F3_30_lr1e-3_dropout2_sgd'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.2,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.2, fc_hidden=(),
    solver='sgd', learning_rate=0.001, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f4['F3_30_lr1e-3_dropout3_sgd'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.3,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.3, fc_hidden=(),
    solver='sgd', learning_rate=0.001, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f4['F3_30_lr5e-4_dropout2_sgd'] = CNN_LSTM_Hybrid(
    cnn_filters=[30], cnn_kernels=[3], cnn_dropout=0.2,
    lstm_hidden=30, lstm_layers=1, lstm_dropout=0.2, fc_hidden=(),
    solver='sgd', learning_rate=0.0005, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

# ============================================================
# 🔥 SGD com arquitetura 50_25 (2 camadas)
# ============================================================

models_hybrid_f4['F3_50_25_lr1e-3_dropout0_sgd'] = CNN_LSTM_Hybrid(
    cnn_filters=[50, 25], cnn_kernels=[3, 3], cnn_dropout=0.0,
    lstm_hidden=25, lstm_layers=2, lstm_dropout=0.0, fc_hidden=(25,),
    solver='sgd', learning_rate=0.001, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f4['F3_50_25_lr5e-4_dropout0_sgd'] = CNN_LSTM_Hybrid(
    cnn_filters=[50, 25], cnn_kernels=[3, 3], cnn_dropout=0.0,
    lstm_hidden=25, lstm_layers=2, lstm_dropout=0.0, fc_hidden=(25,),
    solver='sgd', learning_rate=0.0005, momentum=0.9, epochs=500, batch_size=32, random_state=42
)

models_hybrid_f4['F3_50_25_lr1e-4_dropout0_sgd'] = CNN_LSTM_Hybrid(
    cnn_filters=[50, 25], cnn_kernels=[3, 3], cnn_dropout=0.0,
    lstm_hidden=25, lstm_layers=2, lstm_dropout=0.0, fc_hidden=(25,),
    solver='sgd', learning_rate=0.0001, momentum=0.9, epochs=500, batch_size=32, random_state=42